In [ ]:
!nvidia-smi
import torch

print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(
        f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB"
    )

## Step 1: Clone Repository and Setup Environment

In [ ]:
import os
import getpass
from pathlib import Path

# Configuration
GITHUB_USER = "sattary"
REPO_NAME = "ali_proj"
BRANCH = "fix-review"  # Change if using different branch
PROJECT_DIR = "ali_proj"

print("Enter your GitHub Personal Access Token (PAT):")
PAT = getpass.getpass()
REPO_URL = f"https://{PAT}@github.com/{GITHUB_USER}/{REPO_NAME}.git"

# 1. Clone Repository
if not Path(PROJECT_DIR).exists():
    print(f"Cloning {REPO_NAME} (branch: {BRANCH})...")
    !git clone -b {BRANCH} {REPO_URL}
else:
    print("Repository already cloned. Pulling latest changes...")
    !cd {PROJECT_DIR} && git pull origin {BRANCH}

%cd {PROJECT_DIR}

# 2. Install uv
print("\nInstalling uv...")
!pip install -q uv

# 3. Set MPLBACKEND for Kaggle compatibility
os.environ['MPLBACKEND'] = 'Agg'
print("\nSet MPLBACKEND=Agg for headless environments")

# 4. Sync Dependencies
print("\nSyncing dependencies...")
!uv sync

print("\n✓ Setup complete!")

In [ ]:
!git pull

In [ ]:
/kaggle/input/datasets/rezasattary/config

In [ ]:
!cp ../../input/datasets/rezasattary/config/* runs/

In [ ]:
!ls runs

In [ ]:
# Verify Git Repository
from pathlib import Path
import subprocess

# Check if we're in a git repo
result = subprocess.run(
    ["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True
)
if result.returncode == 0:
    repo_root = result.stdout.strip()
    print(f"✓ Git repository found at: {repo_root}")
    print(f"✓ Current directory: {Path.cwd()}")
else:
    print("Error: Not in a git repository!")
    print(f"Current directory: {Path.cwd()}")
    raise RuntimeError("Git repository not found")

# Show repo status
!git status

## Phase 1: Deterministic Generation (`generate`)

Generate the synthetic interferogram dataset to HDF5 shards. The cryptographic seed guarantees mathematically invariant noise topologies.

In [ ]:
!uv sync

In [ ]:
!ls data

In [ ]:
# Phase 1: Generate Data
!uv run phase-unwrap generate \
    --num-samples 10000 \
    --shard-size 100 \
    --out-dir data/kaggle_full \
    --seed 1337

print("\n✓ Data generation complete!")

## Phase 2: Hyperparameter Optimization (`tune`)

Use Optuna's Bayesian TPE algorithm to isolate the absolute lowest-error configuration. The best configuration is automatically saved to `runs/optuna/best_config.yaml`.

In [ ]:
# Phase 2: Optuna Tune
# Automatically parallelizes across available GPUs
!uv run phase-unwrap tune \
    --use-amp \
    --n-trials 15 \
    --tune-epochs 20 \
    --study-name kaggle_10k_hpo \
    --n-workers 2 \
    --batch-size 64 \
    --data-dir data/kaggle_full/

print("\n✓ Tuning complete!")
print("Best config frozen to: runs/optuna/best_config.yaml")

## Phase 3: Primary Training and Evaluation (`train`)

Train the baseline network to convergence using the frozen `best_config.yaml`.

In [ ]:
!git pull

In [ ]:
# Phase 3: Train Primary Network
import os

config_arg = (
    "--config runs/optuna/best_config.yaml"
    if os.path.exists("runs/optuna/best_config.yaml")
    else ""
)

!uv run phase-unwrap train \
    --use-amp \
    {config_arg} \
    --run-name exp_primary \
    --data-dir data/kaggle_full/ \
    --batch-size 32 \
    --epochs 100

print("\n✓ Primary training complete!")

## Phase 4: Statistical Validation (`multiseed`)

Defend against 'lucky seed' anomalies. Spawns completely independent training convergences using the same frozen configuration, and automatically aggregates the metrics into Mean ± Std.

In [ ]:
%cd ali_proj/

In [ ]:
!git status

In [ ]:
# Phase 4: Statistical Validation
import os

config_arg = (
    "--config runs/optuna/best_config.yaml"
    if os.path.exists("runs/optuna/best_config.yaml")
    else ""
)

!uv run phase-unwrap multiseed \
    --use-amp \
    {config_arg} \
    --run-name exp_multiseed \
    --num-seeds 3 \
    --data-dir data/kaggle_full/ \
    --epochs 100 \
    --batch-size 32

print("\n✓ Multiseed validation complete!")

## Phase 5: Architectural Ablation (`ablation`)

Mathematically prove the necessity of your custom topology by systematically crippling the network. Exports a rigorous LaTeX comparison table.

In [ ]:
!git pull

In [ ]:
!ls runs/

In [ ]:
# Phase 5: Architectural Ablation
import os

config_arg = (
    "--config runs/optuna/best_config.yaml"
    if os.path.exists("runs/optuna/best_config.yaml")
    else ""
)

!uv run phase-unwrap ablation \
    --use-amp \
    {config_arg} \
    --out-table results/tables/ablation.tex \
    --data-dir data/kaggle_full/ \
    --batch-size 32 \
    --epochs 100

print("\n✓ Ablation study complete!")

## Step 6: Zip and Download Results

Run this cell to zip the `runs/` and `results/` directories so you can render them on your local machine.

In [ ]:
import shutil
from IPython.display import FileLink

print("Zipping runs and results...")
shutil.make_archive('training_results', 'zip', 'runs/')
# shutil.make_archive('tables_results', 'zip', 'results/')
print("✓ Done!")
# display(FileLink('training_results.zip'))
# display(FileLink('tables_results.zip'))

In [ ]:
!ls